# MLOps Playbook: MLflow and Airflow

This notebook adds practical MLOps tips for tracking experiments, comparing models, and orchestrating repeatable AI workflows.


## Why these tools matter

- MLflow helps you track experiments, metrics, artifacts, and model versions
- Airflow helps you schedule, orchestrate, and monitor repeatable pipelines

Together they help you move from a notebook demo to a reproducible system.


## MLflow tips

Good MLflow habits:

- log parameters, metrics, and artifacts consistently
- use named experiments for each problem or domain
- compare a baseline against every new run
- keep the evaluation dataset fixed when comparing models
- store the model version that was actually shipped


In [ ]:
experiment = {
    'name': 'agentic-ai-tips',
    'params': {'model': 'small-local', 'prompt_style': 'structured'},
    'metrics': {'accuracy': 0.87, 'latency_ms': 640, 'cost_usd': 0.004},
    'artifacts': ['sample_outputs.json', 'eval_report.md'],
}
experiment


In [ ]:
def compare_runs(baseline: dict, candidate: dict) -> dict:
    return {
        'accuracy_delta': round(candidate['metrics']['accuracy'] - baseline['metrics']['accuracy'], 3),
        'latency_delta_ms': candidate['metrics']['latency_ms'] - baseline['metrics']['latency_ms'],
        'cost_delta': round(candidate['metrics']['cost_usd'] - baseline['metrics']['cost_usd'], 4),
        'winner': 'candidate' if candidate['metrics']['accuracy'] >= baseline['metrics']['accuracy'] else 'baseline',
    }

baseline = {'metrics': {'accuracy': 0.82, 'latency_ms': 720, 'cost_usd': 0.0035}}
candidate = experiment
compare_runs(baseline, candidate)


## Airflow tips

Airflow is best when a process needs scheduling, retries, dependency handling, and visibility over time. For AI systems, use it for tasks like:

- nightly evaluation runs
- dataset refreshes
- model retraining
- report generation
- alerting after failed jobs


In [ ]:
# Airflow-style thinking: tasks are small, explicit, and retryable.
dag = {
    'extract_data': [],
    'validate_data': ['extract_data'],
    'train_model': ['validate_data'],
    'evaluate_model': ['train_model'],
    'publish_model': ['evaluate_model'],
}
dag


## Airflow trick

Treat the pipeline as code, but keep each task narrow. A good task does one thing and emits one clear output.


In [ ]:
def run_task(task_name: str) -> dict:
    return {
        'task': task_name,
        'status': 'success',
        'notes': 'completed with reproducible inputs',
    }

[run_task(task) for task in dag]


## Reproducibility tricks

- version your data
- version your prompts
- version your model
- version your evaluation set
- version your pipeline

If you can’t reproduce it, you can’t really improve it.


In [ ]:
pipeline_manifest = {
    'data_version': 'v1.3',
    'prompt_version': 'p2',
    'model_version': 'm4',
    'eval_set': 'golden-50',
    'orchestrator': 'airflow',
}
pipeline_manifest


## Deployability tip

A strong MLOps setup separates:

- experimentation
- evaluation
- orchestration
- model serving
- monitoring


## Final checklist

Before promoting a model or workflow, confirm:

- the run is tracked
- metrics improved over baseline
- artifacts are saved
- the workflow is retryable
- the output is observable
- rollback is possible
